In [2]:
!pip install -q pycolmap-cuda12 supabase plyfile tqdm pillow
!git clone https://github.com/hbb1/2d-gaussian-splatting --recursive /kaggle/working/2dgs
!sed -i '1i #include <cstdint>' /kaggle/working/2dgs/submodules/diff-surfel-rasterization/cuda_rasterizer/rasterizer_impl.h
!TORCH_CUDA_ARCH_LIST="7.5" pip install --no-build-isolation /kaggle/working/2dgs/submodules/diff-surfel-rasterization /kaggle/working/2dgs/submodules/simple-knn

!git clone --depth 1 https://github.com/nianticlabs/spz.git /kaggle/working/spz
!cmake -S /kaggle/working/spz -B /kaggle/working/spz/build -DSPZ_BUILD_TOOLS=ON
!cmake --build /kaggle/working/spz/build --target ply_to_spz -j4

!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y -qq nodejs
!npm install -g @playcanvas/splat-transform

fatal: destination path '/kaggle/working/2dgs' already exists and is not an empty directory.
Processing ./2dgs/submodules/diff-surfel-rasterization
  Preparing metadata (pyproject.toml) ... done
Processing ./2dgs/submodules/simple-knn
  Preparing metadata (pyproject.toml) ... done
  Created wheel for diff_surfel_rasterization: filename=diff_surfel_rasterization-0.0.1-cp313-cp313-linux_x86_64.whl size=3357694 sha256=0ed2cfc033ce5e7c8a3088a61ce6437d616de98030190786a26d4f90d4d2a42a
  Stored in directory: /tmp/pip-ephem-wheel-cache-t4hxwgg1/wheels/97/de/72/4b16318740b87eea8e101d86e3f97e0931535a79d8ec570492
  Created wheel for simple_knn: filename=simple_knn-0.0.0-cp313-cp313-linux_x86_64.whl size=3220751 sha256=c5fe350a51562991a31d0096810df6980c3b9a68bd4eb63fe0cc9d6a47928f46
  Stored in directory: /tmp/pip-ephem-wheel-cache-t4hxwgg1/wheels/68/dd/01/d7aa83a06f18ebb1cc337e82d7edb5b1f9a80e1c28054cce36
Successfully built diff_surfel_rasterization simple_knn
  Attempting uninstall: simple_knn
 

In [3]:
import os, shutil, subprocess, numpy as np
from datetime import datetime, timezone
from pathlib import Path
from PIL import Image, ImageOps
from plyfile import PlyData, PlyElement
from kaggle_secrets import UserSecretsClient
from supabase import create_client
import pycolmap

os.environ["QT_QPA_PLATFORM"] = "offscreen"

secrets = UserSecretsClient()
sb = create_client(secrets.get_secret("SUPABASE_URL"), secrets.get_secret("SUPABASE_SERVICE_ROLE_KEY"))

# 1. Pick oldest pending job
jobs = sb.table("reconstruction_jobs").select("*").eq("status", "pending").order("created_at").limit(1).execute().data
if not jobs:
    raise RuntimeError("No pending jobs found.")
JOB_ID = jobs[0]["job_id"]
PROJECT_ID = jobs[0]["project_id"]
print(f"Processing job {JOB_ID} for project {PROJECT_ID}")

now = lambda: datetime.now(timezone.utc).isoformat()
sb.table("reconstruction_jobs").update({"status": "running", "stage": "downloading", "started_at": now()}).eq("job_id", JOB_ID).execute()
sb.table("projects").update({"status": "processing"}).eq("project_id", PROJECT_ID).execute()

try:
    # 2. Download & resize images (max 2048px)
    work = Path(f"/kaggle/working/run_{PROJECT_ID}")
    if work.exists(): shutil.rmtree(work)
    img_dir = work / "images"
    img_dir.mkdir(parents=True)

    rows = sb.table("uploaded_images").select("*").eq("project_id", PROJECT_ID).execute().data
    for r in rows:
        path_in_bucket = r["file_path"].removeprefix("project-images/")
        raw = sb.storage.from_("project-images").download(path_in_bucket)
        out = img_dir / Path(r["file_path"]).name
        out.write_bytes(raw)
        with Image.open(out) as im:
            exif = im.info.get("exif")
            im = ImageOps.exif_transpose(im)
            if max(im.size) > 2048:
                im.thumbnail((2048, 2048), Image.LANCZOS)
                im.save(out, quality=92, **({"exif": exif} if exif else {}))

    # 3. COLMAP GPU + Undistort
    sb.table("reconstruction_jobs").update({"stage": "colmap"}).eq("job_id", JOB_ID).execute()
    db, sparse = work / "database.db", work / "sparse"
    sparse.mkdir()
    pycolmap.extract_features(db, img_dir, device=pycolmap.Device.cuda)
    pycolmap.match_exhaustive(db, device=pycolmap.Device.cuda)
    pycolmap.incremental_mapping(db, img_dir, sparse)

    scene = work / "scene_undistorted"
    pycolmap.undistort_images(scene, sparse / "0", img_dir)
    if not (scene / "sparse/0").exists():
        (scene / "sparse/0").mkdir()
        for f in (scene / "sparse").glob("*.bin"):
            shutil.move(str(f), str(scene / "sparse/0" / f.name))

    # 4. 2DGS Training (15k steps)
    sb.table("reconstruction_jobs").update({"stage": "training"}).eq("job_id", JOB_ID).execute()
    STEPS = 15000
    subprocess.run([
        "python", "/kaggle/working/2dgs/train.py",
        "-s", str(scene), "-m", str(work),
        "--iterations", str(STEPS), "--save_iterations", str(STEPS), "--test_iterations", "-1"
    ], check=True)

    # 5. Patch scale_2, reduce to 250k splats & convert PLY -> SPZ
    sb.table("reconstruction_jobs").update({"stage": "exporting"}).eq("job_id", JOB_ID).execute()
    ply = PlyData.read(work / f"point_cloud/iteration_{STEPS}/point_cloud.ply")
    v = ply["vertex"].data
    v_new = np.empty(len(v), dtype=v.dtype.descr + [("scale_2", "f4")])
    for name in v.dtype.names: v_new[name] = v[name]
    v_new["scale_2"] = -20.0
    temp_ply = work / "temp_3dgs.ply"
    PlyData([PlyElement.describe(v_new, "vertex")]).write(temp_ply)

    if len(v) > 500000:
        reduced_ply = work / "reduced_250k.ply"
        subprocess.run(["splat-transform", "-g", "-cpu", str(temp_ply), "--filter-nan", "-d", "500000", str(reduced_ply), "-w"], check=True)
        temp_ply = reduced_ply

    spz_path = work / f"prism_scene_{STEPS:06d}.spz"
    subprocess.run(["/kaggle/working/spz/build/ply_to_spz", str(temp_ply), str(spz_path)], check=True)

    # 6. Upload SPZ & update Supabase
    remote_path = f"{PROJECT_ID}/{spz_path.name}"
    with open(spz_path, "rb") as f:
        sb.storage.from_("project-images").upload(remote_path, f, {"content-type": "application/octet-stream", "upsert": "true"})

    sb.table("reconstructed_models").insert({
        "project_id": PROJECT_ID, "storage_path": remote_path, "file_format": "spz"
    }).execute()
    sb.table("reconstruction_jobs").update({"status": "completed", "stage": "done", "completed_at": now()}).eq("job_id", JOB_ID).execute()
    sb.table("projects").update({"status": "completed"}).eq("project_id", PROJECT_ID).execute()
    print(f"Done! Uploaded {remote_path}")

except Exception as e:
    sb.table("reconstruction_jobs").update({"status": "failed", "completed_at": now()}).eq("job_id", JOB_ID).execute()
    sb.table("projects").update({"status": "failed"}).eq("project_id", PROJECT_ID).execute()
    raise

Processing job c986eff8-b53a-4592-a5a8-2dc734e4b7c0 for project 30a51f71-7342-48de-af7a-32c0e1a086b2


I20261121 04:04:46.692278 133331838559936 feature_extraction.cc:498] === Feature extraction ===
I20261121 04:04:46.692920 133330934683328 sift.cc:758] Creating SIFT GPU feature extractor
I20261121 04:04:46.692996 133330926290624 sift.cc:758] Creating SIFT GPU feature extractor
I20261121 04:04:47.220717 133330917897920 feature_extraction.cc:270] Processed file [1/243]
I20261121 04:04:47.220756 133330917897920 feature_extraction.cc:273]   Name:            1795231367755-3rogd8.jpg
I20261121 04:04:47.220763 133330917897920 feature_extraction.cc:282]   Dimensions:      2048 x 1152
I20261121 04:04:47.220770 133330917897920 feature_extraction.cc:285]   Camera:          #1 - SIMPLE_RADIAL
I20261121 04:04:47.220777 133330917897920 feature_extraction.cc:289]   Focal Length:    1140.40px (Prior)
I20261121 04:04:47.220788 133330917897920 feature_extraction.cc:294]   Features:        7213 (SIFT)
I20261121 04:04:47.296702 133330917897920 feature_extraction.cc:315]   Gravity:         X=0.000, Y=1.000

TypeError: expected str, bytes or os.PathLike object, not int

In [8]:

    # 5. Patch scale_2, reduce to 250k splats & convert PLY -> SPZ
    sb.table("reconstruction_jobs").update({"stage": "exporting"}).eq("job_id", JOB_ID).execute()
    ply = PlyData.read(work / f"point_cloud/iteration_{STEPS}/point_cloud.ply")
    v = ply["vertex"].data
    v_new = np.empty(len(v), dtype=v.dtype.descr + [("scale_2", "f4")])
    for name in v.dtype.names: v_new[name] = v[name]
    v_new["scale_2"] = -20.0
    temp_ply = work / "temp_3dgs.ply"
    PlyData([PlyElement.describe(v_new, "vertex")]).write(temp_ply)

    if len(v) > 250000:
        reduced_ply = work / "reduced_250k.ply"
        subprocess.run(["splat-transform", "-g", "cpu", str(temp_ply), "--filter-nan", "-d", "500000", "--ply-logit-opacity", str(reduced_ply), "-w"], check=True)
        temp_ply = reduced_ply

    spz_path = work / f"prism_scene_{STEPS:06d}.spz"
    subprocess.run(["/kaggle/working/spz/build/ply_to_spz", str(temp_ply), str(spz_path)], check=True)

    # 6. Upload SPZ & update Supabase
    remote_path = f"{PROJECT_ID}/{spz_path.name}"
    with open(spz_path, "rb") as f:
        sb.storage.from_("project-images").upload(remote_path, f, {"content-type": "application/octet-stream", "upsert": "true"})

    sb.table("reconstructed_models").upsert({
        "project_id": PROJECT_ID, "storage_path": remote_path, "file_format": "spz"
    }, on_conflict="project_id").execute()
    sb.table("reconstruction_jobs").update({"status": "completed", "stage": "done", "completed_at": now()}).eq("job_id", JOB_ID).execute()
    sb.table("projects").update({"status": "completed"}).eq("project_id", PROJECT_ID).execute()
    print(f"Done! Uploaded {remote_path}")

splat-transform v3.12.0 (5d5be2b)
▸ [2/2] Output /kaggle/working/run_30a51f71-7342-48de-af7a-32c0e1a086b2/reduced_250k.ply
  ▸ [1/2] Decimate generation
    ▸ reading positions [####################] 0.215s
    ▸ Partitioning
    ▸ computing merge priorities [####################] 1m30.393s
    ▸ Selecting merges
    ▸ merging [####################] 7.800s
  ▸ [2/2] Decimate generation
    ▸ Partitioning
    ▸ computing merge priorities [####################] 40.398s
    ▸ Selecting merges
  ▸ merging [
    · 500K gaussians · 3 SH bands
    ▸ Writing [####################] 2.902s
    ! 222689 gaussians had opacity above 1, clamped to 1 in the standard PLY opacity
  ▸ merging [####################] 3.401s
done in 2m24.799s  [peak cpu=2.10GB]


[SPZ] Loading: /kaggle/working/run_30a51f71-7342-48de-af7a-32c0e1a086b2/reduced_250k.ply
[SPZ] Loading 500000 points
Done! Uploaded 30a51f71-7342-48de-af7a-32c0e1a086b2/prism_scene_015000.spz
